# 005 Router

这是 LangChain Multi-agent 学习线的第五份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent/router

学习目标：

1. 理解 router 是分类和分发步骤，不是持续编排器
2. 区分 deterministic routing 和 model routing
3. 学会用 `Command(goto=...)` 表达单目标路由
4. 学会用 `Send(...)` 表达多目标 fan-out
5. 对比本仓库 Harness 的 planner action 路由

这一讲使用普通 Python 和 LangGraph 的 `Command` / `Send` 类型，不消耗真实模型额度。

## 1. Router 解决什么问题

router 模式适合这样的场景：

```text
用户请求进来
先判断它属于哪个垂直领域
再把请求交给对应 agent / workflow / toolset
```

它解决的是“应该交给谁处理”的问题。

它不是 coordinator。

coordinator 通常还要负责持续计划、综合、验证和下一步控制；router 通常只负责一次或某个阶段的分发。

## 2. Router 和前几种模式的边界

| 模式 | 核心问题 | 典型场景 |
| --- | --- | --- |
| Skill | 当前 agent 需要按需加载什么能力包 | 基金查询、天气查询、文档处理 |
| Subagent | 当前 agent 要委派谁做局部任务 | research / verification |
| Handoff | 后续控制权要交给谁 | 售前转售后、客服转理赔 |
| Router | 这个请求一开始应该进入哪条处理路径 | GitHub / Slack / Notion 三类请求分流 |

一句话：

```text
Router 是入口分流。
Coordinator 是过程管理。
```

In [39]:
from dataclasses import dataclass
from typing import Callable

from langgraph.types import Command, Send


@dataclass(frozen=True)
class RouteDecision:
    agent: str
    query: str
    reason: str


def print_routes(routes: list[RouteDecision]) -> None:
    for route in routes:
        print(route.agent, "<=", route.reason)

## 3. deterministic routing

deterministic routing 是确定性路由。

它通常基于：

- 关键词
- URL / 路径
- 用户所在页面
- 表单类型
- 权限和组织信息

优点：稳定、可解释、便宜。

缺点：覆盖不了复杂语义。

In [40]:
def deterministic_router(user_message: str) -> list[RouteDecision]:
    lowered = user_message.lower()
    routes: list[RouteDecision] = []

    if any(word in lowered for word in ["github", "repo", "pull request", "代码", "仓库"]):
        routes.append(RouteDecision("github_agent", user_message, "命中代码仓库相关词"))

    if any(word in lowered for word in ["notion", "doc", "文档", "知识库"]):
        routes.append(RouteDecision("notion_agent", user_message, "命中文档知识库相关词"))

    if any(word in lowered for word in ["slack", "message", "消息", "讨论"]):
        routes.append(RouteDecision("slack_agent", user_message, "命中团队消息相关词"))

    if not routes:
        routes.append(RouteDecision("general_agent", user_message, "没有命中专用领域，走通用入口"))

    return routes


examples = [
    "帮我检查这个 GitHub repo 的 pull request",
    "去知识库文档里找一下部署说明",
    "把代码仓库和 Slack 讨论都查一下",
    "你好，随便聊聊",
]

for text in examples:
    print("\nuser:", text)
    print_routes(deterministic_router(text))


user: 帮我检查这个 GitHub repo 的 pull request
github_agent <= 命中代码仓库相关词

user: 去知识库文档里找一下部署说明
notion_agent <= 命中文档知识库相关词

user: 把代码仓库和 Slack 讨论都查一下
github_agent <= 命中代码仓库相关词
slack_agent <= 命中团队消息相关词

user: 你好，随便聊聊
general_agent <= 没有命中专用领域，走通用入口


## 4. 用 `Command(goto=...)` 表达单目标路由

如果只需要进入一个目标 agent，可以用 `Command(goto=...)` 表达。

这里先取第一个 route。

真实 LangGraph workflow 里，`goto` 会让图跳转到对应节点。

In [41]:
def route_to_one(user_message: str) -> Command:
    first_route = deterministic_router(user_message)[0]
    return Command(goto=first_route.agent)


single_command = route_to_one("帮我检查这个 GitHub repo 的 pull request")
print(single_command)
print("goto:", single_command.goto)

Command(goto='github_agent')
goto: github_agent


## 5. 用 `Send(...)` 表达多目标 fan-out

有些请求需要查多个来源。

例如：

```text
把代码仓库和 Slack 讨论都查一下
```

这种场景不是简单 `goto` 一个节点，而是 fan-out 到多个 agent。

`Send(node, arg)` 可以表达：

```text
把这个子任务发给某个节点处理。
```

In [42]:
def route_to_many(user_message: str) -> list[Send]:
    return [
        Send(route.agent, {"query": route.query, "reason": route.reason})
        for route in deterministic_router(user_message)
        if route.agent != "general_agent"
    ]


fanout = route_to_many("把代码仓库和 Slack 讨论都查一下")
for item in fanout:
    print(item.node, item.arg)

github_agent {'query': '把代码仓库和 Slack 讨论都查一下', 'reason': '命中代码仓库相关词'}
slack_agent {'query': '把代码仓库和 Slack 讨论都查一下', 'reason': '命中团队消息相关词'}


## 6. 模拟多个 agent 处理 fan-out 结果

下面不启动真实模型，只用普通函数模拟三个垂直 agent。

重点不是模型回答，而是控制流：

```text
router -> 多个 agent -> coordinator synthesis
```

In [44]:
def github_agent(payload: dict) -> dict:
    return {
        "agent": "github_agent",
        "finding": "发现 PR #42 修改了 app/agents/harness.py。",
    }


def notion_agent(payload: dict) -> dict:
    return {
        "agent": "notion_agent",
        "finding": "知识库里部署说明最后更新时间是 2026-05-20。",
    }


def slack_agent(payload: dict) -> dict:
    return {
        "agent": "slack_agent",
        "finding": "Slack 讨论提到 approval 恢复流程需要补测试。",
    }


AGENTS: dict[str, Callable[[dict], dict]] = {
    "github_agent": github_agent,
    "notion_agent": notion_agent,
    "slack_agent": slack_agent,
}


def synthesize(results: list[dict]) -> str:
    lines = [f"- {item['agent']}: {item['finding']}" for item in results]
    return "\n".join(lines)


def run_fanout_router(user_message: str) -> str:
    sends = route_to_many(user_message)
    results = []

    for send in sends:
        handler = AGENTS[send.node]
        results.append(handler(send.arg))

    return synthesize(results)


print(run_fanout_router("把代码仓库和 Slack 讨论都查一下"))

- github_agent: 发现 PR #42 修改了 app/agents/harness.py。
- slack_agent: Slack 讨论提到 approval 恢复流程需要补测试。


## 7. model routing 和 deterministic routing 的取舍

router 可以由模型做，也可以由系统规则做。

| 路由方式 | 优点 | 风险 |
| --- | --- | --- |
| deterministic routing | 稳定、便宜、可解释 | 覆盖不了模糊语义 |
| model routing | 能理解模糊表达 | 需要结构化输出、校验和兜底 |

推荐做法：

```text
明确规则先走确定性路由。
模糊问题再交给模型分类。
模型分类结果必须经过 allowlist 校验。
```

不要让模型返回任意 agent 名称后系统直接执行。

In [45]:
ALLOWED_AGENTS = {"github_agent", "notion_agent", "slack_agent", "general_agent"}


def validate_model_route(agent_name: str) -> str:
    if agent_name not in ALLOWED_AGENTS:
        return "general_agent"
    return agent_name


for model_output in ["github_agent", "delete_database_agent", "slack_agent"]:
    print(model_output, "=>", validate_model_route(model_output))

github_agent => github_agent
delete_database_agent => general_agent
slack_agent => slack_agent


## 8. 和本仓库 Harness planner action 的关系

本仓库 `HarnessChatAgent` 里的 planner action 也有路由含义。

例如：

```text
answer         -> 直接回答
tool           -> 调工具
delegate       -> 委派一个 subagent
delegate_batch -> 并行委派多个 research subagent
```

这和 Router 模式的共同点是：

```text
模型或规则先产生一个受限决策，
系统再根据决策进入对应执行路径。
```

关键点不是“模型想去哪就去哪”，而是：

```text
决策必须在系统允许的 action / agent / tool allowlist 内。
```

## 9. Router 的设计要点

设计 router 时要明确：

1. 允许路由到哪些 agent？
2. 每个 agent 的职责边界是什么？
3. 何时单路由，何时 fan-out？
4. 模型路由结果如何校验？
5. 未命中时走哪里？
6. 多 agent 结果由谁综合？
7. 路由错误如何被发现和回退？

Router 的风险通常不是代码复杂，而是边界没定义清楚。

## 10. 本讲练习

请判断下面场景应该用 deterministic routing、model routing，还是不该用 router：

1. 用户在 `/orders` 页面提问，默认只查订单系统。
2. 用户说“帮我看看昨天大家怎么讨论这个 bug 的”，需要判断是 Slack 还是 GitHub。
3. 用户只是问“Python 的 `lower()` 是什么”。
4. 用户请求同时查 GitHub PR 和 Slack 讨论。

参考答案：

1. deterministic routing
2. model routing 或 deterministic + 模糊兜底
3. 不该用 router，直接回答
4. router fan-out

## 11. 本讲小结

这一讲的核心：

```text
Router 是受控分发，不是自由跳转。
```

你现在应该能判断：

- 什么时候适合 router
- deterministic routing 和 model routing 的取舍
- `Command(goto=...)` 如何表达单目标路由
- `Send(...)` 如何表达 fan-out
- 为什么模型路由必须经过 allowlist 校验

下一讲可以继续进入 Custom workflow。